<a href="https://colab.research.google.com/github/RajeshworM/Yield_Modelling_Automation/blob/main/NASAPOWER_DataProgram.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
# =========================================================
# NASA POWER DAILY DATA DOWNLOAD
# ALL INDIAN DISTRICTS

# =========================================================
# INSTALL PACKAGES
# =========================================================
!pip -q install geopandas requests pyarrow tqdm shapely fiona

# =========================================================
# IMPORTS
# =========================================================
import os
import time
import zipfile
import requests
import pandas as pd
import geopandas as gpd

from tqdm import tqdm

# =========================================================
# OUTPUT DIRECTORIES
# =========================================================
OUTPUT_DIR = "/content/nasa_power_india"

DISTRICT_DIR = os.path.join(OUTPUT_DIR, "district_csvs")

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(DISTRICT_DIR, exist_ok=True)

# =========================================================
# DOWNLOAD INDIA DISTRICT SHAPEFILE
# =========================================================

print("Downloading India district shapefile...")

shp_url = "https://github.com/wmgeolab/geoBoundaries/raw/main/releaseData/gbOpen/IND/ADM2/geoBoundaries-IND-ADM2-all.zip"

zip_path = "/content/india_districts.zip"

r = requests.get(shp_url)

with open(zip_path, "wb") as f:
    f.write(r.content)

extract_dir = "/content/india_districts"

os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print("Shapefile extracted.")

# =========================================================
# FIND SHAPEFILE
# =========================================================

shp_file = None

for root, dirs, files in os.walk(extract_dir):
    for file in files:
        if file.endswith(".shp") and "simplified" not in file:
            shp_file = os.path.join(root, file)

print("Using shapefile:", shp_file)

# =========================================================
# LOAD SHAPEFILE
# =========================================================

gdf = gpd.read_file(shp_file)

print("Total districts:", len(gdf))

# =========================================================
# FIX CENTROID WARNING
# =========================================================

# Reproject to metric CRS
gdf_proj = gdf.to_crs(epsg=3857)

# Compute centroids
centroids = gdf_proj.centroid

# Back to WGS84
centroids = gpd.GeoSeries(centroids, crs=3857).to_crs(epsg=4326)

gdf["lon"] = centroids.x
gdf["lat"] = centroids.y

# =========================================================
# DISTRICT NAME COLUMN
# =========================================================

district_col = "shapeName"

if district_col not in gdf.columns:
    district_col = gdf.columns[0]

print("District column:", district_col)

# =========================================================
# NASA POWER PARAMETERS
# VERIFIED DAILY + AG COMMUNITY
# =========================================================

PARAMETERS = [

    # =====================================================
    # SOLAR
    # =====================================================
    "ALLSKY_SFC_SW_DWN",
    "CLRSKY_SFC_SW_DWN",
    "ALLSKY_SFC_SW_DNI",
    "ALLSKY_SFC_SW_DIFF",
    "TOA_SW_DWN",
    "ALLSKY_SFC_PAR_TOT",
    "CLRSKY_SFC_PAR_TOT",

    # =====================================================
    # TEMPERATURE
    # =====================================================
    "T2M",
    "T2M_MAX",
    "T2M_MIN",
    "T2M_RANGE",
    "T2MDEW",
    "TS",
    "ALLSKY_SFC_LW_DWN",

    # =====================================================
    # HUMIDITY / PRECIP
    # =====================================================
    "RH2M",
    "QV2M",
    "PRECTOTCORR",

    # =====================================================
    # PRESSURE / WIND
    # =====================================================
    "PS",
    "WS2M",
    "WS2M_MAX",
    "WS2M_MIN",
    "WS2M_RANGE",
    "WD2M",

    "WS10M",
    "WS10M_MAX",
    "WS10M_MIN",
    "WS10M_RANGE",
    "WD10M",

    # =====================================================
    # SOIL
    # =====================================================
    "GWETTOP",
    "GWETROOT",
    "GWETPROF",
]

print("Total parameters:", len(PARAMETERS))

# =========================================================
# NASA LIMIT = 20 PARAMETERS MAX
# =========================================================

def chunk_list(lst, size=20):
    for i in range(0, len(lst), size):
        yield lst[i:i + size]

PARAMETER_CHUNKS = list(chunk_list(PARAMETERS, 20))

print("Total parameter chunks:", len(PARAMETER_CHUNKS))

# =========================================================
# NASA API SETTINGS
# =========================================================

BASE_URL = "https://power.larc.nasa.gov/api/temporal/daily/point"

START_DATE = "20000101"

END_DATE = pd.Timestamp.today().strftime("%Y%m%d")

# =========================================================
# REQUEST FUNCTION
# =========================================================

def fetch_power_data(lat, lon, parameter_chunk):

    params = {
        "parameters": ",".join(parameter_chunk),
        "community": "AG",
        "longitude": lon,
        "latitude": lat,
        "start": START_DATE,
        "end": END_DATE,
        "format": "JSON"
    }

    try:

        r = requests.get(BASE_URL, params=params, timeout=120)

        if r.status_code != 200:

            print("ERROR:", r.status_code)
            print(r.text[:500])

            return None

        return r.json()

    except Exception as e:

        print("REQUEST ERROR:", e)

        return None

# =========================================================
# PARSE NASA RESPONSE
# =========================================================

def parse_response(js):

    try:

        param_data = js["properties"]["parameter"]

        dfs = []

        for param_name, values in param_data.items():

            temp = pd.DataFrame({
                "date": list(values.keys()),
                param_name: list(values.values())
            })

            dfs.append(temp)

        merged = dfs[0]

        for d in dfs[1:]:

            merged = merged.merge(
                d,
                on="date",
                how="outer"
            )

        return merged

    except Exception as e:

        print("PARSE ERROR:", e)

        return None

# =========================================================
# MAIN DOWNLOAD LOOP
# =========================================================

all_frames = []

for idx, row in tqdm(gdf.iterrows(), total=len(gdf)):

    district = str(row[district_col]).replace("/", "_")

    lat = row["lat"]
    lon = row["lon"]

    out_csv = os.path.join(
        DISTRICT_DIR,
        f"{district}.csv"
    )

    # =====================================================
    # SKIP IF EXISTS
    # =====================================================

    if os.path.exists(out_csv):

        print("Skipping existing:", district)

        continue

    print(f"\nDownloading: {district}")

    district_df = None

    # =====================================================
    # DOWNLOAD PARAMETER CHUNKS
    # =====================================================

    for chunk_idx, chunk in enumerate(PARAMETER_CHUNKS):

        print(
            f"  Chunk {chunk_idx+1}/{len(PARAMETER_CHUNKS)}"
        )

        success = False

        # Retry attempts
        for attempt in range(5):

            try:

                js = fetch_power_data(
                    lat,
                    lon,
                    chunk
                )

                if js is not None:

                    temp_df = parse_response(js)

                    if temp_df is not None:

                        if district_df is None:

                            district_df = temp_df

                        else:

                            district_df = district_df.merge(
                                temp_df,
                                on="date",
                                how="outer"
                            )

                        success = True

                        break

            except Exception as e:

                print(
                    f"Retry {attempt+1} error:",
                    e
                )

            wait_time = (attempt + 1) * 5

            print(f"Waiting {wait_time}s...")

            time.sleep(wait_time)

        # =================================================
        # CHUNK FAILURE
        # =================================================

        if not success:

            print(
                "FAILED CHUNK:",
                district,
                chunk_idx + 1
            )

        # =================================================
        # AVOID NASA RATE LIMIT
        # =================================================

        time.sleep(0.7)

    # =====================================================
    # SAVE DISTRICT FILE
    # =====================================================

    if district_df is not None:

        district_df["district"] = district
        district_df["latitude"] = lat
        district_df["longitude"] = lon

        cols = [
            "district",
            "latitude",
            "longitude",
            "date"
        ] + [
            c for c in district_df.columns
            if c not in [
                "district",
                "latitude",
                "longitude",
                "date"
            ]
        ]

        district_df = district_df[cols]

        district_df.to_csv(
            out_csv,
            index=False
        )

        all_frames.append(district_df)

        print("Saved:", district)

    else:

        print("FAILED DISTRICT:", district)

# =========================================================
# MERGE ALL DISTRICTS
# =========================================================

print("\nCreating merged dataset...")

if len(all_frames) > 0:

    merged = pd.concat(
        all_frames,
        ignore_index=True
    )

    merged["date"] = pd.to_datetime(
        merged["date"]
    )

    # =====================================================
    # SAVE CSV
    # =====================================================

    merged_csv = os.path.join(
        OUTPUT_DIR,
        "india_nasa_power_daily.csv"
    )

    merged.to_csv(
        merged_csv,
        index=False
    )

    # =====================================================
    # SAVE PARQUET
    # =====================================================

    merged_parquet = os.path.join(
        OUTPUT_DIR,
        "india_nasa_power_daily.parquet"
    )

    merged.to_parquet(
        merged_parquet,
        index=False
    )

    print("\nDONE")

    print("CSV:")
    print(merged_csv)

    print("\nPARQUET:")
    print(merged_parquet)

else:

    print("No data downloaded.")

Shapefile extracted.
Using shapefile: /content/india_districts/geoBoundaries-IND-ADM2.shp
Total districts: 735
District column: shapeName
Total parameters: 31
Total parameter chunks: 2


  0%|          | 0/735 [00:00<?, ?it/s]


Downloading: Ashoknagar
  Chunk 1/2
  Chunk 2/2


  0%|          | 1/735 [00:08<1:39:23,  8.12s/it]

Saved: Ashoknagar

Downloading: Raisen
  Chunk 1/2
  Chunk 2/2


...

100%|█████████▉| 734/735 [1:36:04<00:08,  8.03s/it]

Saved: Tenkasi

Downloading: Gaurella Pendra Marwahi
  Chunk 1/2
  Chunk 2/2


100%|██████████| 735/735 [1:36:12<00:00,  7.85s/it]

Saved: Gaurella Pendra Marwahi

Creating merged dataset...



DONE
CSV:
/content/nasa_power_india/india_nasa_power_daily.csv

PARQUET:
/content/nasa_power_india/india_nasa_power_daily.parquet


In [4]:
from google.colab import files

files.download('/content/nasa_power_india/india_nasa_power_daily.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
from google.colab import files

files.download('/content/nasa_power_india/india_nasa_power_daily.parquet')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [1]:
import pandas as pd

df = pd.read_csv(
    "/content/nasa_power_india/india_nasa_power_daily.csv"
)

print("Total rows:", len(df))
print("Total columns:", len(df.columns))

Total rows: 7009184
Total columns: 35


In [2]:
import pandas as pd

df = pd.read_parquet(
    "/content/nasa_power_india/india_nasa_power_daily.parquet"
)

df.head()

,district,latitude,longitude,date,ALLSKY_SFC_SW_DWN,CLRSKY_SFC_SW_DWN,ALLSKY_SFC_SW_DNI,ALLSKY_SFC_SW_DIFF,TOA_SW_DWN,ALLSKY_SFC_PAR_TOT,...,WS2M_RANGE,WD2M,WS10M,WS10M_MAX,WS10M_MIN,WS10M_RANGE,WD10M,GWETTOP,GWETROOT,GWETPROF
0,Ashoknagar,24.608475,77.876485,2000-01-01,17.43,17.47,-999.0,4.52,23.17,7.35,...,1.01,83.0,2.43,3.42,0.85,2.57,83.6,0.44,0.59,0.59
1,Ashoknagar,24.608475,77.876485,2000-01-02,16.16,16.43,-999.0,4.49,23.22,7.15,...,1.17,88.9,2.09,3.46,0.52,2.94,87.0,0.43,0.59,0.59
2,Ashoknagar,24.608475,77.876485,2000-01-03,16.72,16.87,-999.0,4.35,23.27,7.29,...,2.12,78.2,2.63,3.28,0.51,2.77,87.4,0.42,0.58,0.59
3,Ashoknagar,24.608475,77.876485,2000-01-04,17.48,17.58,-999.0,3.60,23.33,7.49,...,1.61,78.9,2.62,3.30,1.14,2.16,96.6,0.41,0.58,0.59
4,Ashoknagar,24.608475,77.876485,2000-01-05,18.29,18.23,-999.0,3.61,23.40,7.54,...,1.71,66.2,2.58,3.30,0.87,2.43,72.1,0.40,0.58,0.59


In [3]:
for district, subdf in df.groupby("district"):

    out = f"/content/{district}.xlsx"

    subdf.to_excel(out, index=False)

KeyboardInterrupt: 

In [ ]:
df[df["district"] == "Indore"].head()

In [4]:
# =========================================================
# ZIP ALL DISTRICT CSV FILES
# =========================================================

import shutil
from google.colab import files

source_folder = "/content/nasa_power_india/district_csvs"

zip_path = "/content/india_district_csvs.zip"

print("Creating ZIP file...")

shutil.make_archive(
    zip_path.replace(".zip", ""),
    'zip',
    source_folder
)

print("ZIP created:")
print(zip_path)

# =========================================================
# DOWNLOAD ZIP
# =========================================================

files.download(zip_path)

Creating ZIP file...
ZIP created:
/content/india_district_csvs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>